# Generate and verify multi-sublattice supercells
This bounded example generates diboride, MAX-211 and Heusler cells with fixed compositions. It uses two MPI ranks and requires the native optimizer, but not ATAT or icet. Lattice parameters are illustrative. Small finite cells and short searches need not reach zero SRO.


In [ ]:
from pathlib import Path
import sys
import tempfile
import json
import numpy as np
from ase.spacegroup import crystal

# Works from a checkout or an installed m_scraps package.
for root in (Path.cwd(), Path.cwd().parent):
    if (root / "python/m_scraps").is_dir():
        sys.path.insert(0, str(root / "python"))
        break
from m_scraps import analyze_structure
output = Path(tempfile.mkdtemp(prefix="m_scraps_analysis_"))
print("Results:", output)


## Prototype definitions

* **(Ti,Zr)B₂:** AlB₂ type, P6/mmm (191), M at 1a and B at 2d;
  [AFLOW prototype index](https://aflow.org/p/prototype_index.html), AB2_hP3_191_a_d.
* **(Ti,Zr)₂(Al,Si)C:** MAX-211, P6₃/mmc (194), M at 4f,
  A at 2d, C at 2a. These positions are described in
  [the Ti₂AX structural study](https://www.nature.com/articles/s41598-022-17539-8).
* **(Co,Ni)₂(Mn,Fe)Al:** L2₁, Fm-3m (225), X at 8c, Y at 4a,
  Z at 4b; equivalent to the
  [AFLOW L2₁ prototype](https://aflow.org/p/AB2C_cF16_225_a_c_b-001/)
  under an origin shift exchanging 4a and 4b.

Each conventional cell is repeated 2×2×2. The `spacegroup_kinds` array tracks
which supplied basis representative generated each site; it is independent of
the chemical symbols assigned afterward.


In [ ]:
prototypes = {
    "diboride": dict(symbols=["Ti", "B"], basis=[(0,0,0), (1/3,2/3,.5)],
        spacegroup=191, cellpar=[3.1,3.1,3.3,90,90,120],
        labels=["M", "B"], allowed={"M": ["Ti", "Zr"], "B": ["B"]}),
    "max211": dict(symbols=["Ti", "Al", "C"], basis=[(1/3,2/3,.08), (1/3,2/3,.75), (0,0,0)],
        spacegroup=194, cellpar=[3.05,3.05,13.6,90,90,120],
        labels=["M", "A", "C"], allowed={"M": ["Ti", "Zr"], "A": ["Al", "Si"], "C": ["C"]}),
    "heusler": dict(symbols=["Co", "Mn", "Al"], basis=[(.25,.25,.25), (0,0,0), (.5,.5,.5)],
        spacegroup=225, cellpar=[5.95,5.95,5.95,90,90,90],
        labels=["X", "Y", "Z"], allowed={"X": ["Co", "Ni"], "Y": ["Mn", "Fe"], "Z": ["Al"]}),
}
expected = {
    "diboride": {"Ti":4, "Zr":4, "B":16},
    "max211": {"Ti":16, "Zr":16, "Al":8, "Si":8, "C":16},
    "heusler": {"Co":32, "Ni":32, "Mn":16, "Fe":16, "Al":32},
}


In [ ]:
from collections import Counter
from m_scraps import run_sqs_from_atoms
results = {}
for name, spec in prototypes.items():
    primitive = crystal(spec["symbols"], spec["basis"], spacegroup=spec["spacegroup"], cellpar=spec["cellpar"])
    species = [s for alphabet in spec["allowed"].values() for s in alphabet]
    counts = [expected[name][s] for s in species]
    sublattices, spectators = [], []
    for gid, label in enumerate(spec["labels"]):
        basis_sites = np.flatnonzero(primitive.arrays["spacegroup_kinds"] == gid).tolist()
        indices = [species.index(s) for s in spec["allowed"][label]]
        if len(indices) == 1:
            spectators.extend((i, indices[0]) for i in basis_sites)
        else:
            sublattices.append((gid, basis_sites, indices))
    optimized = run_sqs_from_atoms(primitive, species=species, element_counts=counts,
        cell_dim=(2,2,2), work_dir=output/name, ranks=2, rng_seed=42,
        nests=8, mcstep=32, total_iter=2, auto_mcstep=0, extra_lines="GITER 1\n",
        max_shellnum=2, weights=[1,1], sublattices=sublattices,
        spectator_basis_species=spectators)
    reference = primitive.repeat((2,2,2))
    labels = [spec["labels"][int(i)] for i in reference.arrays["spacegroup_kinds"]]
    config = dict(site_labels=labels, allowed=spec["allowed"], counts={
        label: {s: expected[name][s] for s in alphabet} for label, alphabet in spec["allowed"].items()})
    report = analyze_structure(optimized, reference=reference, sublattices=config, cutoff=8, shells=2)
    assert dict(Counter(optimized.get_chemical_symbols())) == expected[name]
    optimized.write(output/f"{name}_optimized.cif")
    (output/f"{name}_verified.json").write_text(json.dumps(report, indent=2, allow_nan=False))
    results[name] = report
    mixed = [r for r in report["sro"] if not r["fixed_pair"] and r["alpha"] is not None]
    print(name, len(optimized), "largest reported |alpha|:", max(abs(r["alpha"]) for r in mixed))


The analyzer reports shells separately for each sublattice pair; native pair-mode shell indices use its configured global distance shells. The largest reported SRO above therefore describes the stated analysis scope, not a reconstruction of the optimizer fitness. For identical multi-figure target metrics, use the frozen figure benchmark and `compare-atat` workflow.
